In [2]:
from typing import Literal, Union
import openai
import dotenv
from pydantic import BaseModel


class TypeA(BaseModel):
    type: Literal["a"]


class TypeB(BaseModel):
    type: Literal["b"]


class UnionOfStuff(BaseModel):
    a: Union[TypeA, TypeB]


dotenv.load_dotenv()

client = openai.OpenAI()

response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": "Hello pick a 80% of the time a TypeA and 20% of the time a TypeB!",
        }
    ],
    response_format=UnionOfStuff,
)

In [3]:
response.choices[0].message.content

'{"a":{"type":"a"}}'

In [4]:
from abc import ABC, abstractmethod


class Primitive(ABC):
    @abstractmethod
    async def execute(self):
        """
        Execute the primitive.

        Subclasses must implement this method.
        """
        pass

    def guidelines(self):
        """
        Optionally provide guidelines for this primitive.
        Subclasses may override this method if guidelines are available.
        """
        return None

In [13]:
import inspect


def primitive_to_dict(primitive_obj):
    """
    Given a Primitive instance, returns a dict with:
      - "guideline": the guidelines text (if a guidelines() method exists)
      - "inputs": a list of tuples (input_name, type_name) for each parameter of execute (other than self)
    """
    result = {}

    # If the object has a guidelines method, call it and store its result.
    guidelines_func = getattr(primitive_obj, "guidelines", None)
    if callable(guidelines_func):
        result["guideline"] = guidelines_func()

    # Use inspect.signature to get the parameters of the execute method.
    execute_func = getattr(primitive_obj, "execute", None)
    inputs = {}
    if callable(execute_func):
        sig = inspect.signature(execute_func)
        # Loop over parameters, skipping "self"
        for name, param in sig.parameters.items():
            if name == "self":
                continue
            # Determine the type annotation, if any.
            if param.annotation is inspect.Parameter.empty:
                type_name = None
            else:
                # If the annotation is a type, get its __name__
                if isinstance(param.annotation, type):
                    type_name = param.annotation.__name__
                else:
                    # Otherwise, use the string representation.
                    type_name = str(param.annotation)
            inputs[name] = type_name
    result["inputs"] = inputs

    return result

In [14]:
# Example primitive without guidelines.
class ServeGlass(Primitive):
    def __init__(self):
        super().__init__()

    def guidelines(self):
        return "Serve a glass of water. The glass has to be in sight."

    async def execute(self, distance: float):
        # Gentle handover motion for glass.
        print("Serving glass.")
        return "Served glass.", True

In [15]:
primitive_to_dict(ServeGlass())

{'guideline': 'Serve a glass of water. The glass has to be in sight.',
 'inputs': {'distance': 'float'}}

In [8]:
from baml_client.type_builder import TypeBuilder


def build_union_of_primitives(primitives_list: list[dict]):
    """
    Given a list of dictionaries representing primitive definitions,
    build and return a dynamic BAML type representing a union of these primitives.

    Each primitive dict is expected to have:
      - "guideline": Optional string with guidelines for the primitive.
      - "inputs": A list of tuples (input_name, type_name). If no type is provided,
                  the type value is None.

    The returned type (UnionOfPrimitives) can be used as the type of the next_task field
    in a VisionAgentOutput-like dynamic type.
    """
    tb = TypeBuilder()

    # Define a dynamic type for an individual input.
    # Each input has a "name" and an optional "type".
    primitive_input = tb.add_class("PrimitiveInput")
    primitive_input.add_property("name", tb.string())
    primitive_input.add_property("type", tb.string().optional()).description(
        "Type name of the input (or None if not provided)"
    )

    # Define a dynamic class to represent the union of primitive variants.
    union_primitives = tb.add_class("UnionOfPrimitives")

    # For each primitive dictionary, create a dynamic variant.
    for idx, prim in enumerate(primitives_list):
        variant_name = f"PrimitiveVariant{idx}"
        variant = tb.add_class(variant_name)

        # If a guideline exists, add it as a property.
        if prim.get("guideline"):
            variant.add_property("guideline", tb.string()).description(
                "Guidelines for this primitive"
            )

        # Add the "inputs" property. In our schema it is a list of PrimitiveInput objects.
        variant.add_property("inputs", primitive_input.type().list()).description(
            "Inputs for the execute function"
        )

        # (Optionally, if you want to “bake in” default values from prim,
        #  you might add metadata—but here we simply build the schema.)

        # Add this variant as an optional property of the union.
        union_primitives.add_property(
            variant_name, variant.type().optional()
        ).description(f"Primitive variant: {variant_name}")

    return union_primitives.type()


def build_dynamic_vision_agent_output(primitives_list: list[dict]):
    """
    Build a dynamic BAML type equivalent to VisionAgentOutput, whose fields mirror:

        stop_current_task: bool
        observation: str
        thoughts: str
        new_goal: Optional[str]
        next_task: Optional[UnionOfPrimitives]
        anticipation: Optional[str]
        to_tell_user: Optional[str]

    The next_task field is built as a dynamic union of primitives using the
    build_union_of_primitives() function.
    """
    tb = TypeBuilder()
    union_primitives = build_union_of_primitives(primitives_list)

    vision_agent_output = tb.add_class("VisionAgentOutput")
    vision_agent_output.add_property("stop_current_task", tb.bool())
    vision_agent_output.add_property("observation", tb.string())
    vision_agent_output.add_property("thoughts", tb.string())
    vision_agent_output.add_property("new_goal", tb.string().optional())
    vision_agent_output.add_property(
        "next_task", union_primitives.optional()
    ).description("Next task as a union of primitives")
    vision_agent_output.add_property("anticipation", tb.string().optional())
    vision_agent_output.add_property("to_tell_user", tb.string().optional())

    return vision_agent_output.type()


# --- Example Usage ---

if __name__ == "__main__":
    # Imagine you have collected primitive definitions like this:
    primitives = [
        {
            "guideline": (
                "Can be called in front of a cardboard glass, to let the arm grab it. \n"
                "If the glass is not cardboard, the primitive should not be called.\n"
                "The glass has to be handed by a human, it cannot be picked up from the ground."
            ),
            "inputs": [("dummy_arg", None)],
        },
        {
            # This primitive did not define guidelines.
            "guideline": None,
            "inputs": [],
        },
    ]

    # Build the dynamic VisionAgentOutput type.
    VisionAgentOutputDynamic = build_dynamic_vision_agent_output(primitives)

    # At runtime you would pass the TypeBuilder (which now holds your dynamic types)
    # in the options to your BAML function. For example:
    #
    #   from baml_client.async_client import b
    #   tb = TypeBuilder()  # (and your above modifications would have been done on tb)
    #   result = await b.YourBamlFunction(..., { "tb": tb })
    #
    # Here we simply print the dynamic type definition:
    print(VisionAgentOutputDynamic)

In [12]:
VisionAgentOutputDynamic

In [2]:
from baml_utils import vision_agent
import base64
import dotenv

dotenv.load_dotenv()

with open("test.jpg", "rb") as file:
    base64_str = base64.b64encode(file.read()).decode("utf-8")

res = await vision_agent(base64_str)

[2025-02-12T05:59:38Z INFO  baml_events] Function VisionAgent:
    Client: OpenAIClient (gpt-4o-2024-08-06) - 6418ms. StopReason: stop. Tokens(in/out): 1036/123
    ---PROMPT---
    [chat] user: Decide what to do next with this image:<image_placeholder base64>.
    
    Consider the guidelines of the primitives carefully before picking one to do (if any).
    
    Answer in JSON using this schema:
    {
      stop_current_task: bool,
      observation: string,
      thoughts: string,
      new_goal: string or null,
      // Optional next task as a union of primitives
      next_task: {
        // Serve a glass of water. The glass has to be in sight.
        guideline: string,
        inputs: {
          distance: float,
        },
      } or {
        // Grab a bottle with a specified color. The bottle must be within reach.
        guideline: string,
        inputs: {
          bottle_color: string,
        },
      } or null,
      anticipation: string or null,
      to_tell_user: str

In [3]:
res

VisionAgentOutput(stop_current_task=True, observation='The image shows a receipt from Forbes Cafe for a Pho Noodle Soup with double protein, fully paid.', thoughts='The image is simply a receipt without any actionable items for me to perform.', new_goal=None, next_task=None, anticipation='No further action is required as the image does not indicate any tasks related to the primitives.', to_tell_user="No task can be performed based on this image; it's a receipt.")